In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 1000

wiek = np.random.randint(18, 70, n)
dochód = np.random.randint(3000, 25000, n)
wizyty_www = np.random.randint(1, 50, n)
czas_na_stronie = np.random.uniform(0.5, 20.0, n)

# Sztuczna logika biznesowa:
# większy dochód, więcej wizyt i dłuższy czas na stronie
# zwiększają szansę zakupu
score = (
    0.02 * wiek
    + 0.00015 * dochód
    + 0.12 * wizyty_www
    + 0.35 * czas_na_stronie
    - 8
)

prob = 1 / (1 + np.exp(-score))
kupi = (np.random.rand(n) < prob).astype(int)

df = pd.DataFrame({
    "wiek": wiek,
    "dochód": dochód,
    "wizyty_www": wizyty_www,
    "czas_na_stronie": czas_na_stronie,
    "kupi": kupi
})

df.to_csv("customers.csv", index=False)

print("Zapisano plik customers.csv")
print(df.head())
print("\nRozkład klas:")
print(df["kupi"].value_counts())

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==============================
# Konfiguracja
# ==============================
CSV_PATH = "customers.csv"
MODEL_PATH = "customer_purchase_model.pt"
SCALER_PATH = "customer_purchase_scaler.joblib"

BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# CPU / GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Używane urządzenie:", device)

# ==============================
# 1. Wczytanie danych z CSV
# ==============================
df = pd.read_csv(CSV_PATH)

print("Podgląd danych:")
print(df.head())

print("\nInformacje o danych:")
print(df.info())

# ==============================
# 2. Definicja kolumn
# ==============================
feature_columns = ["wiek", "dochód", "wizyty_www", "czas_na_stronie"]
target_column = "kupi"

X = df[feature_columns].values
y = df[target_column].values

# ==============================
# 3. Podział train/test
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# ==============================
# 4. Skalowanie danych
# ==============================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, SCALER_PATH)
print(f"\nZapisano scaler do: {SCALER_PATH}")

# ==============================
# 5. Zamiana na tensory
# ==============================
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

# BCEWithLogitsLoss oczekuje floatów 0.0 / 1.0
y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==============================
# 6. Definicja modelu
# ==============================
class CustomerClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)

model = CustomerClassifier(input_dim=len(feature_columns)).to(device)
print("\nModel:")
print(model)

# ==============================
# 7. Loss i optimizer
# ==============================
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ==============================
# 8. Trening
# ==============================
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] - loss: {epoch_loss:.4f}")

# ==============================
# 9. Wykres loss
# ==============================
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.title("Loss podczas treningu")
plt.xlabel("Epoka")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

# ==============================
# 10. Ewaluacja
# ==============================
model.eval()

all_probs = []
all_preds = []
all_true = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)

        logits = model(batch_X)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        all_probs.extend(probs.cpu().numpy().flatten())
        all_preds.extend(preds.cpu().numpy().flatten())
        all_true.extend(batch_y.numpy().flatten())

accuracy = accuracy_score(all_true, all_preds)

print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification report:")
print(classification_report(all_true, all_preds, digits=4))

print("Confusion matrix:")
print(confusion_matrix(all_true, all_preds))

# ==============================
# 11. Zapis modelu .pt
# ==============================
torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": len(feature_columns),
    "feature_columns": feature_columns
}, MODEL_PATH)

print(f"\nZapisano model do: {MODEL_PATH}")

In [ ]:
import joblib
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

MODEL_PATH = "customer_purchase_model.pt"
SCALER_PATH = "customer_purchase_scaler.joblib"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class CustomerClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)

# Wczytanie checkpointa
checkpoint = torch.load(MODEL_PATH, map_location=device)

input_dim = checkpoint["input_dim"]
feature_columns = checkpoint["feature_columns"]

model = CustomerClassifier(input_dim=input_dim).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Wczytanie scalera
scaler = joblib.load(SCALER_PATH)

# Nowe dane do predykcji
new_customers = pd.DataFrame([
    {"wiek": 25, "dochód": 4500, "wizyty_www": 3, "czas_na_stronie": 1.2},
    {"wiek": 39, "dochód": 12000, "wizyty_www": 15, "czas_na_stronie": 8.5},
    {"wiek": 52, "dochód": 22000, "wizyty_www": 27, "czas_na_stronie": 14.0},
])

X_new = new_customers[feature_columns].values
X_new_scaled = scaler.transform(X_new)
X_new_tensor = torch.tensor(X_new_scaled, dtype=torch.float32).to(device)

with torch.no_grad():
    logits = model(X_new_tensor)
    probs = torch.sigmoid(logits).cpu().numpy().flatten()
    preds = (probs >= 0.5).astype(int)

new_customers["prawdopodobienstwo_zakupu"] = probs
new_customers["predykcja_kupi"] = preds

print(new_customers)